# 01 - Agent Architecture
The agent runs a ReAct loop: **THINK** (generate or fix?) -> **ACT** (call Gemini, run tests) -> **OBSERVE** (pass/fail) -> **DECIDE** (stop or iterate).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.agent.code_agent import CodeGenerationAgent

## Offline demo (no API)
A fake generator returns a buggy first draft, then a fix, showing the self-repair loop.

In [2]:
class FakeGen:
    def generate_code(self, task, error_context='', previous_code=''):
        return 'def solution(xs):\n    return sorted(xs)\n' if error_context else 'def solution(xs):\n    return xs\n'
    def generate_tests(self, code, task):
        return 'def test_a():\n    assert solution([2, 1]) == [1, 2]\n'

r = CodeGenerationAgent(generator=FakeGen()).run('sort a list')
print(r['success'], 'iterations:', r['iterations'])

2026-10-13 07:39:23,101 INFO iter=1 action=generate outcome=CONTINUE


2026-10-13 07:39:25,080 INFO iter=2 action=fix outcome=SUCCESS


True iterations: 2


## Live run with Gemini

In [3]:
agent = CodeGenerationAgent()
r = agent.run('Sort a list of dictionaries by their age field, handling missing age keys')
print('success:', r['success'], '| iterations:', r['iterations'])
print(r['code'])

2026-10-13 07:40:13,043 INFO iter=1 action=generate outcome=CONTINUE


2026-10-13 07:41:11,851 INFO iter=2 action=fix outcome=CONTINUE


2026-10-13 07:41:27,228 INFO iter=3 action=fix outcome=SUCCESS


success: True | iterations: 3
def solution(people, reverse=False):
    has_age = []
    no_age = []
    for person in people:
        if person.get("age") is not None:
            has_age.append(person)
        else:
            no_age.append(person)

    return sorted(has_age, key=lambda x: x["age"], reverse=reverse) + no_age


sort_by_age = solution


## Per-iteration history

In [4]:
for h in agent.get_conversation_history():
    print('iteration', h['iteration'], '->', 'PASS' if h['result'].get('success') else 'FAIL')
    print(h['code'], '\n')

iteration 1 -> FAIL
Bob', 'age': None}`, `x.get('age')` is `None`.
If `x` is `{'name': 'Bob', 'age': 25}`, `x.get('age')` is `25`.

`key=lambda x: (x.get('age') is None, x.get('age') if x 

iteration 2 -> FAIL
def sort_by_age(people, reverse=False):
    return sorted(
        people,
        key=lambda x: (
            x.get("age") is None,
            x.get("age") if x.get("age") is not None else 0,
        ),
        reverse=reverse,
    ) 

iteration 3 -> PASS
def solution(people, reverse=False):
    has_age = []
    no_age = []
    for person in people:
        if person.get("age") is not None:
            has_age.append(person)
        else:
            no_age.append(person)

    return sorted(has_age, key=lambda x: x["age"], reverse=reverse) + no_age


sort_by_age = solution 

